# DINOv3 ViT-L/16 phase2 → SSL → TTA

Pipeline:
1. Wczytaj **phase2 checkpoint**
2. **Generuj pseudo-labele** dla testu (próg pewności 0.95)
3. **SSL: 1 epoka** treningu na `train2 ∪ pseudo_high` (wagi startują z phase2)
4. Zapisz model po SSL
5. **TTA 5 wariantów** (orig, hflip, bright_up, bright_down, hflip+bright_up) → submission

## Ważne decyzje (zgodne z phase2)
- `crop_bbox` **bez paddingu** (jak w phase2, nie jak w kodzie referencyjnym z paddingiem)
- Focal Loss **bez wag klas** 
- `state_dict` zapisany przez `bare(model)` (bez prefiksu `module.`)
- AMP `bf16` na T4 (wspierane), inaczej `fp16`
- Pseudo-labele z `VAL_TRANSFORM` (bez TTA — jak w kodzie referencyjnym)
- Wagi czworo treningowe nie są zamrażane (SSL to fine-tuning całego modelu z niskim LR)

## Konfiguracja Kaggle
- Accelerator: **GPU T4×2**
- Internet: **ON**
- **+ Add Input**: competition + dataset/model z `dinov3_vitl16_phase2.pt`


In [ ]:
!pip install -q --upgrade timm
import timm; print(f'timm version: {timm.__version__}')

In [ ]:
# ── Znajdź checkpoint ────────────────────────────────────────────────────────
!find /kaggle/input -name 'dinov3_vitl16_phase2.pt' 2>/dev/null
!ls /kaggle/input/

In [ ]:
import ast, copy, os, re, time
from pathlib import Path
from PIL import Image

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, ConcatDataset, WeightedRandomSampler
from torchvision import transforms
import torchvision.transforms.functional as TF

from sklearn.metrics import f1_score

import timm

torch.backends.cudnn.benchmark        = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32       = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
NUM_GPUS = torch.cuda.device_count() if DEVICE.type == 'cuda' else 0
USE_DATA_PARALLEL = NUM_GPUS >= 2

print(f'Device: {DEVICE}  |  GPU count: {NUM_GPUS}')
if NUM_GPUS > 0:
    for i in range(NUM_GPUS):
        print(f'  [{i}] {torch.cuda.get_device_name(i)}')
    bf16_ok = torch.cuda.is_bf16_supported()
    AMP_DTYPE = torch.bfloat16 if bf16_ok else torch.float16
    print(f'AMP dtype: {AMP_DTYPE}')

In [ ]:
# ── Konfiguracja ──────────────────────────────────────────────────────────────
CLASS_NAMES = {
    0: 'Lateral_lying_left',
    1: 'Lateral_lying_right',
    2: 'Sitting',
    3: 'Standing',
    4: 'Sternal_lying',
}
NUM_CLASSES = len(CLASS_NAMES)
FLIP_LABEL_MAP = {0: 1, 1: 0, 2: 2, 3: 3, 4: 4}

BASE_DIR = Path('/kaggle/input/competitions/multi-view-pig-posture-recognition/multiview_pig_posture_recognition')
TRAIN2_IMGS = BASE_DIR / 'train2_images'
TEST_IMGS   = BASE_DIR / 'test_images'

DINOV3_REPO = 'vit_large_patch16_dinov3.lvd1689m'
IMG_SIZE    = 224
BATCH_SIZE  = 32 if USE_DATA_PARALLEL else 16

# ── SSL hyperparams ──────────────────────────────────────────────────────────
SSL_THRESHOLD = 0.95
SSL_EPOCHS    = 1
SSL_LR        = 2e-5

# ── Focal Loss (jak w phase2) ────────────────────────────────────────────────
FOCAL_GAMMA = 1.5

# ⬇⬇⬇ USTAW ŚCIEŻKĘ DO PHASE2 CHECKPOINTU ⬇⬇⬇
PHASE2_CKPT = '/kaggle/input/dino-vit-ph2/pytorch/default/1/dinov3_vitl16_phase2.pt'

OUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('.')
SSL_CKPT_PATH    = str(OUT_DIR / 'dinov3_vitl16_phase2_SSL.pt')
SUBMISSION_PATH  = str(OUT_DIR / 'submission_T2_2phase_SSL_TTA.csv')

assert os.path.exists(PHASE2_CKPT), f'Nie znaleziono checkpointu: {PHASE2_CKPT}'
print(f'Phase2 ckpt: {PHASE2_CKPT}')
print(f'SSL ckpt   → {SSL_CKPT_PATH}')
print(f'Submission → {SUBMISSION_PATH}')
print(f'SSL: prog={SSL_THRESHOLD}, epok={SSL_EPOCHS}, lr={SSL_LR}')

## Wczytanie train2 + test (jak w treningu phase2)

In [ ]:
def parse_camera_meta(image_id):
    m = re.match(r'(pen\d+)_(orb|tur)_(cam\d+)_', str(image_id))
    return (m.group(1), m.group(2), m.group(3)) if m else ('unknown', 'unknown', 'unknown')

def add_camera_cols(df):
    df['pen']      = df['image_id'].apply(lambda x: parse_camera_meta(x)[0])
    df['cam_type'] = df['image_id'].apply(lambda x: parse_camera_meta(x)[1])
    df['cam_num']  = df['image_id'].apply(lambda x: parse_camera_meta(x)[2])
    df['camera']   = df['pen'] + '_' + df['cam_type'] + '_' + df['cam_num']
    return df

train2 = pd.read_csv(BASE_DIR / 'train2.csv')
train2['source']      = 'train2'
train2['bbox_parsed'] = train2['bbox'].apply(ast.literal_eval)
train2['class_name']  = train2['class_id'].map(CLASS_NAMES)
train2 = add_camera_cols(train2)

test = pd.read_csv(BASE_DIR / 'test.csv')
test['source']      = 'test'
test['bbox_parsed'] = test['bbox'].apply(ast.literal_eval)
test = add_camera_cols(test)

print(f'Train2: {len(train2):,} instancji')
print(f'Test:   {len(test):,} instancji')
print('\nRozkład klas (train2):')
for cname, cnt in train2['class_name'].value_counts().items():
    print(f'  {cname:25s}: {cnt:,}')

In [ ]:
# ── Image loading + crop bez paddingu (jak w phase2) ─────────────────────────
def load_image(image_id, source):
    folder = {'train2': TRAIN2_IMGS, 'test': TEST_IMGS}[source]
    return Image.open(folder / image_id).convert('RGB')

def crop_bbox(image, bbox):
    img_w, img_h = image.size
    x, y, w, h   = map(float, bbox)
    x1 = max(0, int(round(x)))
    y1 = max(0, int(round(y)))
    x2 = min(img_w, int(round(x + w)))
    y2 = min(img_h, int(round(y + h)))
    return image.crop((x1, y1, max(x2, x1+1), max(y2, y1+1)))

## Augmentacje per kamera + transformacje (jak w phase2)

In [ ]:
# ── Augmentacje per kamera (kopia 1:1 z phase2 treningu) ─────────────────────
def aug_pen2_tur_cam1(img):
    img = TF.hflip(img)
    img = TF.adjust_saturation(img, saturation_factor=0.8)
    return TF.adjust_brightness(img, brightness_factor=0.95)

def aug_pen1_tur_cam2(img):
    img = TF.hflip(img)
    img = TF.adjust_brightness(img, brightness_factor=1.35)
    return TF.adjust_saturation(img, saturation_factor=0.9)

def aug_pen2_orb_cam1(img):
    img = TF.hflip(img)
    img = TF.adjust_brightness(img, brightness_factor=0.6)
    img = TF.adjust_contrast(img, contrast_factor=1.5)
    arr = np.array(img).astype(float)
    arr[:,:,0] = (arr[:,:,0] * 0.85).clip(0, 255)
    arr[:,:,1] = (arr[:,:,1] * 1.10).clip(0, 255)
    arr[:,:,2] = (arr[:,:,2] * 0.80).clip(0, 255)
    shifted = Image.fromarray(arr.clip(0,255).astype(np.uint8))
    grey    = np.array(TF.to_grayscale(shifted, num_output_channels=3)).astype(float)
    return Image.fromarray((0.65*arr + 0.35*grey).clip(0,255).astype(np.uint8))

def aug_pen2_tur_cam2(img):
    img = TF.adjust_brightness(img, brightness_factor=1.3)
    return TF.adjust_saturation(img, saturation_factor=0.85)

def aug_pen2_orb_cam2(img):
    img  = TF.adjust_brightness(img, brightness_factor=0.60)
    img  = TF.adjust_contrast(img, contrast_factor=1.5)
    arr  = np.array(img).astype(float)
    grey = np.array(TF.to_grayscale(img, num_output_channels=3)).astype(float)
    return Image.fromarray((0.65*arr + 0.35*grey).clip(0,255).astype(np.uint8))

CAMERA_AUG_FN = {
    'pen2_tur_cam1': (aug_pen2_tur_cam1, True),
    'pen1_tur_cam2': (aug_pen1_tur_cam2, True),
    'pen2_orb_cam1': (aug_pen2_orb_cam1, True),
    'pen2_tur_cam2': (aug_pen2_tur_cam2, False),
    'pen2_orb_cam2': (aug_pen2_orb_cam2, False),
}

# ── Transformacje (normalizacja ImageNet — używana przez DINOv3) ─────────────
class AddGaussianNoise:
    def __init__(self, std=0.02, p=0.15):
        self.std, self.p = std, p
    def __call__(self, t):
        if torch.rand(1).item() < self.p:
            t = torch.clamp(t + torch.randn_like(t) * self.std, 0., 1.)
        return t

MU  = [0.485, 0.456, 0.406]
STD_LIST = [0.229, 0.224, 0.225]

BASE_TRANSFORM = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    AddGaussianNoise(std=0.02, p=0.15),
    transforms.Normalize(MU, STD_LIST),
])

VAL_TRANSFORM = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MU, STD_LIST),
])

# ── Dla pseudo-labelowanych test'ów: rotation + jitter (z kodu referencyjnego) ──
SSL_TRANSFORM = transforms.Compose([
    transforms.RandomRotation(degrees=90),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.15, hue=0.05),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    AddGaussianNoise(std=0.02, p=0.15),
    transforms.Normalize(MU, STD_LIST),
])

print('Transformacje gotowe ✓')

## Datasety (jak w phase2)

In [ ]:
class PigDatasetCameraAug(Dataset):
    """Dla każdej próbki dodaje wersję augmentowaną pod kamerę treningową
    (z swapem etykiet L↔R przy flipowaniu poziomym). is_train=True duplikuje."""
    def __init__(self, df, camera_aug_fn, base_transform, is_train=True):
        self.camera_aug_fn  = camera_aug_fn
        self.base_transform = base_transform
        df = df.reset_index(drop=True)
        self.df = df
        self.samples = []
        for idx, row in df.iterrows():
            cam   = row.get('camera', 'unknown')
            label = int(row['class_id'])
            self.samples.append((idx, False, label))
            if is_train and cam in camera_aug_fn:
                _, does_flip = camera_aug_fn[cam]
                self.samples.append((idx, True,
                                     FLIP_LABEL_MAP[label] if does_flip else label))

    def __len__(self):  return len(self.samples)

    def __getitem__(self, i):
        row_i, do_aug, label = self.samples[i]
        row  = self.df.iloc[row_i]
        img  = load_image(row['image_id'], row['source'])
        crop = crop_bbox(img, row['bbox_parsed'])
        if do_aug:
            aug_fn, _ = self.camera_aug_fn[row['camera']]
            crop = aug_fn(crop)
        return self.base_transform(crop), label


class PigTestDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df.reset_index(drop=True)
        self.transform = transform
    def __len__(self):  return len(self.df)
    def __getitem__(self, i):
        row  = self.df.iloc[i]
        img  = load_image(row['image_id'], row['source'])
        crop = crop_bbox(img, row['bbox_parsed'])
        return self.transform(crop), str(row['row_id'])


class PseudoDataset(Dataset):
    """Test images z pseudo-etykietami, augmentacja SSL_TRANSFORM (rotation+jitter)."""
    def __init__(self, df, transform):
        self.df = df.reset_index(drop=True)
        self.transform = transform
    def __len__(self):  return len(self.df)
    def __getitem__(self, i):
        row  = self.df.iloc[i]
        img  = load_image(row['image_id'], row['source'])
        crop = crop_bbox(img, row['bbox_parsed'])
        return self.transform(crop), int(row['class_id'])

print('Datasety ✓')

## Architektura modelu (kopia 1:1 z treningu phase2)

In [ ]:
class AttentionPool(nn.Module):
    def __init__(self, embed_dim, num_heads=8, dropout=0.1):
        super().__init__()
        self.query = nn.Parameter(torch.randn(1, 1, embed_dim) * 0.02)
        self.attn  = nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout, batch_first=True)
        self.norm  = nn.LayerNorm(embed_dim)
    def forward(self, tokens):
        B = tokens.size(0)
        q = self.query.expand(B, -1, -1)
        out, _ = self.attn(q, tokens, tokens)
        return self.norm(out.squeeze(1))


class DINOv3Classifier(nn.Module):
    def __init__(self, model_name, num_classes, num_heads=8, mlp_hidden=512, dropout=0.2):
        super().__init__()
        self.backbone = timm.create_model(model_name, pretrained=True, num_classes=0)
        self.embed_dim = self.backbone.embed_dim
        self.num_prefix_tokens = getattr(self.backbone, 'num_prefix_tokens', 5)
        self.attn_pool = AttentionPool(self.embed_dim, num_heads=num_heads, dropout=dropout)
        self.classifier = nn.Sequential(
            nn.Linear(self.embed_dim, mlp_hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_hidden, num_classes),
        )
    def forward(self, pixel_values):
        tokens = self.backbone.forward_features(pixel_values)
        patch_tokens = tokens[:, self.num_prefix_tokens:, :]
        pooled = self.attn_pool(patch_tokens)
        return self.classifier(pooled)


def bare(m):
    return m.module if hasattr(m, 'module') else m


class FocalLoss(nn.Module):
    def __init__(self, gamma: float = 1.5, alpha=None):
        super().__init__()
        self.gamma = gamma
        if alpha is not None:
            self.register_buffer('alpha', alpha.float())
        else:
            self.alpha = None
    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, weight=self.alpha, reduction='none')
        pt = torch.exp(-ce)
        focal = (1.0 - pt) ** self.gamma * ce
        return focal.mean()

## Wczytanie modelu phase2

In [ ]:
print(f'Buduję architekturę DINOv3 ({DINOV3_REPO})...')
model = DINOv3Classifier(DINOV3_REPO, NUM_CLASSES).to(DEVICE)

print(f'Wczytuję wagi z: {PHASE2_CKPT}')
ckpt = torch.load(PHASE2_CKPT, map_location=DEVICE)
state = ckpt['model_state_dict'] if 'model_state_dict' in ckpt else ckpt
if any(k.startswith('module.') for k in state.keys()):
    state = {k.replace('module.', '', 1): v for k, v in state.items()}

missing, unexpected = model.load_state_dict(state, strict=True)
print(f'  missing: {len(missing)}  unexpected: {len(unexpected)}')
if 'best_f1' in ckpt:
    print(f'  Train best F1 (z checkpointu): {ckpt["best_f1"]:.4f}')

if USE_DATA_PARALLEL:
    model = nn.DataParallel(model)
    print(f'DataParallel włączony na {NUM_GPUS} GPU')

model.eval()
print('Model gotowy ✓')

## Krok 1: Generowanie pseudo-etykiet (bez TTA, sam model)

In [ ]:
def generate_pseudo_labels(current_model, test_df, transform):
    print('Generowanie pseudo-etykiet dla zbioru testowego...')
    current_model.eval()
    loader = DataLoader(
        PigTestDataset(test_df, transform),
        batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True
    )

    confs_all, preds_all = [], []
    t0 = time.time()
    with torch.no_grad():
        for bi, (images, _) in enumerate(loader):
            images = images.to(DEVICE, non_blocking=True)
            with torch.amp.autocast('cuda', dtype=AMP_DTYPE, enabled=(DEVICE.type=='cuda')):
                outputs = current_model(images)
            probs = F.softmax(outputs.float(), dim=1)
            confs, preds = probs.max(dim=1)
            confs_all.extend(confs.cpu().numpy().tolist())
            preds_all.extend(preds.cpu().numpy().tolist())

            if (bi+1) % 10 == 0 or (bi+1) == len(loader):
                print(f'\r  Batch {bi+1}/{len(loader)}  ({time.time()-t0:.0f}s)',
                      end='', flush=True)
    print()

    pseudo_df = test_df.copy()
    pseudo_df['class_id']   = preds_all
    pseudo_df['confidence'] = confs_all
    return pseudo_df

pseudo_all = generate_pseudo_labels(model, test, VAL_TRANSFORM)

print(f'\nWszystkich pseudo-labeli: {len(pseudo_all):,}')
print(f'Mediana confidence: {pseudo_all["confidence"].median():.4f}')
print(f'\nRozkład confidence:')
for thr in [0.5, 0.7, 0.8, 0.9, 0.95, 0.99]:
    n = (pseudo_all['confidence'] >= thr).sum()
    pct = 100 * n / len(pseudo_all)
    print(f'  ≥ {thr:.2f}: {n:>6,}  ({pct:5.1f}%)')

## Krok 2: SSL — 1 epoka na train2 ∪ pseudo (próg 0.95)

In [ ]:
# ── Train dataset z augmentacjami per kamera (jak w phase2 treningu) ─────────
train_dataset = PigDatasetCameraAug(train2, CAMERA_AUG_FN, BASE_TRANSFORM, is_train=True)
train_labels  = [s[2] for s in train_dataset.samples]
print(f'Train (po camera-aug): {len(train_dataset):,} próbek')

# ── Filtrujemy pseudo-labele po progu pewności ───────────────────────────────
pseudo_high = pseudo_all[pseudo_all['confidence'] >= SSL_THRESHOLD].copy()
print(f'\nPseudo z conf ≥ {SSL_THRESHOLD}: {len(pseudo_high):,} '
      f'({100*len(pseudo_high)/len(pseudo_all):.1f}%)')
print('Rozkład klas w pseudo-high:')
for cid, cnt in pseudo_high['class_id'].value_counts().sort_index().items():
    print(f'  [{cid}] {CLASS_NAMES[cid]:25s}: {cnt:,}')

pseudo_ds   = PseudoDataset(pseudo_high, SSL_TRANSFORM)
combined_ds = ConcatDataset([train_dataset, pseudo_ds])

# Etykiety łączone (do weighted sampler) — train już ma camera-aug-swapped labels,
# pseudo są nieswapowane.
all_labels_combined = list(train_labels) + pseudo_high['class_id'].tolist()
print(f'\nCombined dataset: {len(combined_ds):,} próbek')

# ── Weighted sampler dla zbalansowanego treningu ─────────────────────────────
counts_comb = np.bincount(all_labels_combined, minlength=NUM_CLASSES)
sample_w = (1.0 / np.maximum(counts_comb, 1))[np.array(all_labels_combined)]
sampler = WeightedRandomSampler(
    weights=torch.DoubleTensor(sample_w),
    num_samples=len(sample_w),
    replacement=True,
)

ssl_loader = DataLoader(
    combined_ds, batch_size=BATCH_SIZE, sampler=sampler,
    num_workers=2, pin_memory=True, persistent_workers=True,
)
print(f'SSL batches: {len(ssl_loader)}')

# ── Focal Loss BEZ wag (jak w phase2) ────────────────────────────────────────
criterion_ssl = FocalLoss(gamma=FOCAL_GAMMA, alpha=None)

# ── Optimizer + AMP ──────────────────────────────────────────────────────────
optimizer_ssl = torch.optim.AdamW(model.parameters(), lr=SSL_LR, weight_decay=1e-5)
scheduler_ssl = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer_ssl, T_max=SSL_EPOCHS, eta_min=1e-6
)
scaler = torch.amp.GradScaler('cuda', enabled=(AMP_DTYPE == torch.float16))

print(f'\nSSL: {SSL_EPOCHS} epoka, lr={SSL_LR}, focal γ={FOCAL_GAMMA}, alpha=None')

In [ ]:
# ── SSL training loop ────────────────────────────────────────────────────────
best_f1_ssl   = 0.0
best_state_ssl = None

for epoch in range(SSL_EPOCHS):
    model.train()
    all_preds, all_labels_tr = [], []
    running_loss = 0.0
    t0 = time.time()
    n_batches = len(ssl_loader)

    for bi, (images, labels) in enumerate(ssl_loader):
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        optimizer_ssl.zero_grad(set_to_none=True)

        with torch.amp.autocast('cuda', dtype=AMP_DTYPE, enabled=(DEVICE.type=='cuda')):
            outputs = model(images)
            loss = criterion_ssl(outputs, labels)

        if AMP_DTYPE == torch.float16:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer_ssl)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer_ssl)
            scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer_ssl.step()

        running_loss += loss.item()
        all_preds.extend(outputs.argmax(dim=1).detach().cpu().tolist())
        all_labels_tr.extend(labels.cpu().tolist())

        bar_len = int(30*(bi+1)/n_batches)
        bar = '█' * bar_len + '░' * (30 - bar_len)
        print(f'\r  |{bar}| {bi+1}/{n_batches}  loss={loss.item():.4f}',
              end='', flush=True)

    print()
    scheduler_ssl.step()
    tr_f1 = f1_score(all_labels_tr, all_preds, average='macro', zero_division=0)
    avg_loss = running_loss / n_batches
    print(f'  Epoka {epoch+1}/{SSL_EPOCHS} ({(time.time()-t0)/60:.1f}m) '
          f'| Train F1: {tr_f1:.4f}  |  Loss: {avg_loss:.4f}')

    if tr_f1 > best_f1_ssl:
        best_f1_ssl = tr_f1
        best_state_ssl = copy.deepcopy(bare(model).state_dict())

# Ładujemy najlepszy stan (przy 1 epoce = ten ostatni)
bare(model).load_state_dict(best_state_ssl)
print(f'\nSSL best train F1: {best_f1_ssl:.4f}')

# ── Zapis ────────────────────────────────────────────────────────────────────
torch.save({
    'model_state_dict': best_state_ssl,
    'best_f1':          best_f1_ssl,
    'ssl_threshold':    SSL_THRESHOLD,
    'ssl_epochs':       SSL_EPOCHS,
    'ssl_lr':           SSL_LR,
    'n_pseudo_used':    len(pseudo_high),
    'phase':            'phase2+SSL',
}, SSL_CKPT_PATH)
print(f'Zapisano → {SSL_CKPT_PATH}')

## Krok 3: TTA 5 wariantów na modelu po SSL

Augmentacje robione na **znormalizowanych tensorach na GPU**: orig, hflip, bright_up, bright_down, hflip+bright_up.

Dla wariantów z `hflip` zamieniamy `probs[:, 0] ↔ probs[:, 1]` (Lateral_lying_left ↔ right) przed uśrednianiem.

In [ ]:
# ── 6. Inferencja z TTA ──────────────────────────────────────────────────────
print('\n' + '='*50)
print(' Inferencja z TTA (po SSL)')
print('='*50)

model.eval()

# Permutacja indeksów dla fliponutego obrazu: 0↔1, reszta zostaje
FLIP_PERM = torch.tensor([FLIP_LABEL_MAP[i] for i in range(NUM_CLASSES)], device=DEVICE)

# Mean/Std jako tensory na GPU
MEAN = torch.tensor([0.485, 0.456, 0.406], device=DEVICE).view(1, 3, 1, 1)
STD  = torch.tensor([0.229, 0.224, 0.225], device=DEVICE).view(1, 3, 1, 1)

def denorm(x): return x * STD + MEAN
def renorm(x): return (x - MEAN) / STD

def tta_identity(x):           return x, False
def tta_hflip(x):              return torch.flip(x, dims=[-1]), True

def tta_brightness_up(x):
    img = denorm(x); img = torch.clamp(img * 1.15, 0.0, 1.0)
    return renorm(img), False

def tta_brightness_down(x):
    img = denorm(x); img = torch.clamp(img * 0.85, 0.0, 1.0)
    return renorm(img), False

def tta_hflip_brightness_up(x):
    img = denorm(x); img = torch.clamp(img * 1.15, 0.0, 1.0)
    img = torch.flip(img, dims=[-1])
    return renorm(img), True

TTA_TRANSFORMS = [
    ('orig',            tta_identity),
    ('hflip',           tta_hflip),
    ('bright_up',       tta_brightness_up),
    ('bright_down',     tta_brightness_down),
    ('hflip+bright_up', tta_hflip_brightness_up),
]

print(f'Liczba wariantów TTA: {len(TTA_TRANSFORMS)} '
      f'({", ".join(name for name, _ in TTA_TRANSFORMS)})')

In [ ]:
# ── DataLoader testowy ──────────────────────────────────────────────────────
test_loader_tta = DataLoader(
    PigTestDataset(test, VAL_TRANSFORM),
    batch_size=BATCH_SIZE, shuffle=False,
    num_workers=2, pin_memory=True,
)

all_row_ids   = []
all_preds_tta = []
n_test_batches = len(test_loader_tta)

t0 = time.time()
with torch.no_grad():
    for batch_i, (images, row_ids) in enumerate(test_loader_tta):
        images = images.to(DEVICE, non_blocking=True)
        probs_sum = torch.zeros(images.size(0), NUM_CLASSES, device=DEVICE)

        for _, tta_fn in TTA_TRANSFORMS:
            aug_images, swap_lr = tta_fn(images)
            with torch.amp.autocast('cuda', dtype=AMP_DTYPE, enabled=(DEVICE.type=='cuda')):
                logits = model(aug_images)
            probs = F.softmax(logits.float(), dim=1)
            if swap_lr:
                probs = probs.index_select(1, FLIP_PERM)
            probs_sum += probs

        probs_mean = probs_sum / len(TTA_TRANSFORMS)
        preds = probs_mean.argmax(dim=1).cpu().numpy()

        all_row_ids.extend(list(row_ids))
        all_preds_tta.extend(preds.tolist())

        if (batch_i + 1) % 10 == 0 or (batch_i + 1) == n_test_batches:
            print(f'\r  [TTA] Batch {batch_i+1}/{n_test_batches}  '
                  f'({time.time()-t0:.0f}s)', end='', flush=True)
print(f'\n\nUkończono TTA w {(time.time()-t0)/60:.1f} min')

In [ ]:
# ── Zapis submission ─────────────────────────────────────────────────────────
submission_tta = pd.DataFrame({'row_id': all_row_ids, 'class_id': all_preds_tta})
sample_sub = pd.read_csv(BASE_DIR / 'sample_submission.csv')

assert set(submission_tta['row_id']) == set(sample_sub['row_id']), 'Niezgodne row_id!'
assert submission_tta['class_id'].between(0, 4).all(), 'Błędne class_id!'

submission_tta = (submission_tta.set_index('row_id')
                                 .reindex(sample_sub['row_id'])
                                 .reset_index())

submission_tta.to_csv(SUBMISSION_PATH, index=False)
print(f'Gotowe! Submission TTA zapisany jako: {SUBMISSION_PATH}\n')
print('Rozkład predykcji TTA:')
for cid, cnt in sorted(submission_tta['class_id'].value_counts().items()):
    pct = 100 * cnt / len(submission_tta)
    bar = '█' * int(pct / 2)
    print(f'  [{cid}] {CLASS_NAMES[cid]:25s}: {cnt:>6,}  ({pct:5.1f}%)  {bar}')

print(f'\nModel SSL:    {SSL_CKPT_PATH}')
print(f'Submission:   {SUBMISSION_PATH}')